**Cordis 2014 · 完整流程与对象图解**

[课程目录](../../README.md) · [环境与运行](../../SETUP.md)


# 本课导览

**2014 是当前 Cordis 系列的收束课。我们用 2013 的完整 ReportDesk，把分课读过的机制连接成一套框架认识。**

前面的课程沿应用需求逐步深入：装插件、提供服务、声明依赖、处理事件、清理资源、更新配置、传递调用上下文，再到服务隔离。本课回头解释这些能力怎样共同工作：**谁保存什么状态，谁决定插件能否运行，一次业务调用经过哪些对象，环境变化后谁负责重建与清理。**

**代码承接关系。** `app/` 的 9 个业务文件完整沿用 2013；两课 `materials/cordis/src/` 的 9 个内核文件也一致，均来自固定上游提交。本课有自己的实际文件、依赖与运行目录，阅读和修改都在本课进行。入口 `main.ts` 保留上一课的业务顺序，只增加四个阶段的框架状态快照；新增的 `shared/observe-framework.ts` 与画图脚本用于解释对象关系，业务不依赖它们。

| 本课继续使用的内容 | 位置 | 阅读重点 |
| --- | --- | --- |
| 两组工作区，各自拥有 report/storage；alpha 关闭后 beta 继续生成报告 | [main.ts](examples/01-reportdesk/main.ts)、[workspace.ts](app/workspace.ts) | 用熟悉的应用恢复完整运行现场 |
| 服务生成、格式环绕、内存保存、文件写出与完成通知 | [report-service.ts](app/report-service.ts)、[report-batch.ts](app/report-batch.ts) | 把业务步骤连接到框架职责 |
| 运行时对象观察与教学图解 | [observe-framework.ts](shared/observe-framework.ts)、[图解索引](materials/README.md) | 按需核对图中的对象；无需先读快照工具的实现 |

本课的理解增量集中在两个方面：一是贯通 **Context → 管理服务 → Runtime/Fiber → 业务服务与回调**，二是用同一套对象关系解释 **正常调用、依赖变化、配置更新与最终关闭**。这些是阅读目标，沿用阅读与观察，不要求另交复述、绘图或作答。

**阅读安排**：M1 框架对象与职责 40 分钟；M2 装载与业务调用 35 分钟；M3 数据与生命周期 40 分钟；M4 系列归纳 10 分钟；缓冲 25 分钟，共 150 分钟。阅读、理解和笔记约 105 分钟，运行与按需调试约 20 分钟。熟悉的应用代码只看摘录；遇到具体关系不清楚时再跳到实现。


# 源码与阅读导航

以下材料按正文就近使用，无需按文件列表通读。内核均为本课固定提交，源码摘录用于阅读；00、01 是唯一运行入口。

| 资料 | 指定范围与停止点 | 用途 |
| --- | --- | --- |
| [S1 · 承接 2013 的完整 ReportDesk 应用](examples/01-reportdesk/main.ts) | M1.1 恢复 main/workspace 装配；M2.2–M2.3 只读 batch、report、格式与观察者摘录；M3 回到 dispose；到各段解释完的结果停止 | 保持同一业务底座，把应用使用方式连接到框架职责 |
| [S2 · 根对象、插件身份与装载记录](materials/cordis/src/registry.ts) | M1.2：context 构造器与 extend、Reflect 的 mixin、registry.plugin、Fiber 构造器；到共享管理服务、Runtime 与装载归属建立处停止 | 理解框架对象的创建、持有与访问入口 |
| [S3 · 服务登记、查找与清理](materials/cordis/src/reflect.ts) | M2.2：handler.get 的成功读取；M3：provide/_getImpl/notify，读到登记、按键查找与相关依赖刷新 | 连接调用位置、实现记录和依赖传播 |
| [S4 · 格式调用与完成通知](materials/cordis/src/events.ts) | M2.3：on/register、waterfall、emit；到本例两个格式回调的返回与两个观察者执行停止 | 区分返回值加工、通知以及清理责任 |
| [S5 · 装载、依赖恢复与清理](materials/cordis/src/fiber.ts) | M2.1：构造器、_checkImpl/_refresh/_reload/await；M3：effect/_unload/update/restart 与 dispose 闭包；只读正文选段连接处 | 串起依赖准备、执行周期、状态变化与释放责任 |
| [S6 · 方法调用与定义处 Context](materials/cordis/src/utils.ts) | M2.2：createTraceable/createShadow/createShadowMethod 与 Reflect 成功读取；以 generate 中的调用信息和 storage 依赖为停止点 | 同时保留调用位置与方法代码的依赖归属 |
| [S7 · 五张图与本课资料索引](materials/README.md) | M1 对象图；M2 装载流程、业务时序；M3 登记数据与生命周期；按对应正文阅读，不要求绘图 | 从同一组对象理解结构、流程、数据与变化 |
| [S9 · Service 构造与实际登记](materials/cordis/src/service.ts) | M2.1：constructor 中保存 ctx、配置 tracker、reflect.provide；到实例登记结束停止 | 把应用 new Service 与框架的实现登记接起来 |

Cordis 上游版本为 4.0.0-rc.10，提交 c261dd2c13d65b1994adb1cdc10d7dba61542523（2026-10-10 UTC）；本课应用、资料与源码核对日期为 2026-10-24。


# 沿场景理解源码

先在本课目录执行 `npm ci` 和 `npm run check`。Notebook 使用 JavaScript（jslab），按 00 → 01 运行；终端使用 `npm run demo`，两者加载同一份本课源码。各阶段的观察说明紧邻代码。

运行生成的报告、状态快照和声明位于本课 `runs/`。重跑会替换同一用途的结果；需要比较时先另行保存。只修改本课文件即可，不依赖其他课程的安装目录。VS Code 请单独打开本课目录，再选择对应调试配置。


## M1 从完整应用看 Cordis 的对象结构

约 40 分钟，含一次运行与自由记录。本段把 2001–2005 的装配和服务、2011–2013 的上下文与隔离放在同一幅结构中。先认出应用中的实际对象，再解释框架为它们保存哪些记录。


### M1.1 从上一课的完整代码恢复现场

ReportDesk 接收固定文本，生成 Markdown 报告。根部安装通知、统计、引用格式和页尾格式；alpha、beta 各安装自己的 report/storage。两组先各处理 lamp，关闭 alpha 后 beta 再处理 bottle，最后释放其余应用插件。

[main.ts](examples/01-reportdesk/main.ts) 的主线摘录如下，省略 import、读取输入、快照与 finally 的兜底清理；`jobs` 来自 `sampleJobs()`，分别为 lamp 和 bottle。它与 2013 的业务主线一致：

```ts
const app = new Context();
const sharedPlugins = [notificationPlugin, statisticsPlugin, quoteFormat, footerFormat]
    .map((plugin) => app.plugin(plugin));
const alphaCtx = app.isolate("report").isolate("storage").extend({ workspaceName: "alpha" });
const betaCtx = app.isolate("report").isolate("storage").extend({ workspaceName: "beta" });
const alpha = alphaCtx.plugin(workspacePlugin);
const beta = betaCtx.plugin(workspacePlugin);

for (const plugin of sharedPlugins) await plugin;
await alpha;
await beta;
await alpha.ctx.plugin(reportBatch, { jobs: [jobs[0]!] });
await beta.ctx.plugin(reportBatch, { jobs: [jobs[0]!] });
await alpha.dispose();
await beta.ctx.plugin(reportBatch, { jobs: [jobs[1]!] });
await beta.dispose();
// 完整入口在 finally 中释放共享插件，并处理异常时仍未关闭的工作区。
```

`app`、`alphaCtx`、`betaCtx` 是 **Context：从哪个位置访问框架与业务能力**。`alpha`、`beta` 是插件装载返回的 Fiber 包装对象：关联这次装载的 ctx，可等待当前装载过程，也可调用 dispose。`alpha.ctx.plugin(...)` 使调用插件归 alpha 这次装载管理。

[workspacePlugin](app/workspace.ts) 说明一组服务从哪里来：

```ts
export async function workspacePlugin(ctx: Context): Promise<void> {
    const storage = ctx.plugin(storageProvider);
    await storage;
    const report = ctx.plugin(reportProvider, { excerptLength: 8 });
    await report;
}
```

storageProvider 执行 `new StorageService(ctx)`，reportProvider 执行 `new ReportService(ctx, config)`。两组分别运行这些代码，创建两组实例；`isolate()` 为相同名字选择不同登记键。**实例的创建、实例的可见范围、装载的清理归属分别由不同代码决定**，后面的对象图会把这三种关系拆开。


### M1.2 Context、管理服务与装载记录各保存什么

从 [context.ts 的构造器](materials/cordis/src/context.ts) 恢复 2002 的根装配。以下为关键语句摘录，省略初始映射、日志服务和最后的初始化清理：

```ts
const self = new Proxy<this>(this, ReflectService.handler);
this.root = self;
this.fiber = new Fiber(self, {}, Object.create(null), null, () => []);
this.reflect = new ReflectService(self);
this.registry = new RegistryService(self);
this.events = new EventsService(self);
return self;
```

因此 `app` 是根 Context 的代理入口；根部先创建特殊的根 Fiber（uid 为 0、runtime 为 null），再创建共享管理服务。管理服务有自己的状态，Context 让插件从当前访问位置使用这些状态：

| 对象 | 保存或负责什么 | 本例怎样使用 |
| --- | --- | --- |
| Context | 当前 Fiber、普通元信息，以及继承得到的服务名映射 | workspaceName 标识入口；extend 派生访问位置 |
| RegistryService | 插件回调到 Runtime 的登记表 | plugin 装载 workspacePlugin、reportProvider、reportBatch |
| ReflectService | 属性访问规则、服务名到实际实现的查找与登记 | ctx.report、ctx.storage；provide 登记服务 |
| EventsService | 事件名到监听记录的表，以及派发方式 | report/format 环绕处理、report/completed 通知 |
| Fiber | 一次插件装载的 config、inject、状态、本轮依赖与释放函数 | 决定何时执行插件，重载或关闭时释放其资源 |

`app.plugin()` 的方法实现就在共享 RegistryService 上。[ReflectService 构造器与 mixin](materials/cordis/src/reflect.ts) 将 registry.plugin、events.on 等能力暴露到 Context；读取混入方法时绑定相应服务视图，保留当前 Context。因而 `alphaCtx.plugin()` 与 `betaCtx.plugin()` 使用同一登记表，却把不同装载入口传给 Fiber。2002 的方法转发在这里连接到了实际插件的归属。

再看 [registry.ts 的 plugin](materials/cordis/src/registry.ts) 中的关键语句。`plugin`、`config` 是调用参数；此处 `this.ctx` 是本次调用的 Context。`getOuterStack` 来自该方法的默认参数，用于错误栈定位。省略参数校验、匿名名称处理与返回包装：

```ts
const callback = this.resolve(plugin);
let runtime = this._internal.get(callback);
if (!runtime) {
    const name = plugin.name;
    runtime = { name, callback, fibers: new DisposableList(), Config: plugin.Config };
    this._internal.set(callback, runtime);
}
const fiber = new Fiber(this.ctx, config, Inject.resolve(plugin.inject), runtime, getOuterStack);
```

**Runtime 按插件回调身份归类，Fiber 按每次装载分别建立。** 对象插件 reportProvider 的身份来自其 apply 函数；函数插件 workspacePlugin 使用函数自身。相同插件代码能对应同一 Runtime 中的多个 Fiber，每个 Fiber 又能通过自己的 apply 创建业务实例。因此“一个 Runtime”“一个 Fiber”“一个 ReportService”是三种不同的计数。

[Fiber 构造器](materials/cordis/src/fiber.ts) 为非根装载执行 `parent.extend({ fiber: this })`，得到传给插件的 ctx，并将该次装载的释放函数登记到 `parent.fiber`。这里只读到 ctx 来源和父级资源登记；依赖如何驱动 apply 留到 M2.1。


### M1.3 运行 01，把图中的身份落到现场

首次运行请先执行 `npm ci` 和 `npm run check`。cwd 使用本课一级目录；Notebook 按 00→01 执行。终端的 `npm run demo` 与 Notebook 共用 [main.ts](examples/01-reportdesk/main.ts)，无需再跑一套独立业务例子。

这次运行用于核对两类对象：**装载记录是否增减，业务实例是否沿用**。每次启动新进程，只重建本课 `runs/reportdesk/`、保留最新结果；需要比较多次结果时，先保存希望保留的报告与快照。

00 只载入共用入口：


In [ ]:
// 00 · ReportDesk：初始化本课 Notebook 入口。
var reportdesk2014 = require(process.cwd().endsWith("2014.Cordis-ReportDesk-Full-Flow")
    ? "./shared/notebook.cjs"
    : "./2014.Cordis-ReportDesk-Full-Flow/shared/notebook.cjs");


01 用同一个应用运行一次。只看“实例是否变化、内存里有哪些报告”；完整对象数据保存到文件，不展开成控制台日志。


In [ ]:
// 01 · ReportDesk：完整流程与对象图解。
reportdesk2014.run();


预期新进程中的主要结果：

| 时刻 | alpha | beta |
| --- | --- | --- |
| 各写 lamp 后 | report/storage 实例 1；ids 为 lamp | report/storage 实例 2；ids 为 lamp |
| 关闭 alpha 后 | 服务登记撤下 | 实例仍为 2；lamp 仍在 |
| beta 再写 bottle 后 | 保持关闭 | 实例仍为 2；ids 为 lamp、bottle |

表前的“汇总打印位置”指向 [saveObservations](shared/observe-workspaces.ts)，每行“采集位置”指向 main.ts 的 `snapshot()` 调用。两者分别是记录状态与统一展示的位置，行号由当前源码取得。可以直接在采集行设断点，不必搜索所有 console.log。

三份 Markdown 文件与 observations.json 位于 runs/reportdesk；新增的 framework.json 记录图中 Runtime、Fiber、服务登记项及两类监听数量。它由 [observe-framework.ts](shared/observe-framework.ts) 只读采集内部字段，不参与业务。关掉服务后文件仍留在磁盘，内存存储与这些文件是两份不同的产物。


### M1.4 对象图：同时看三种关系

图把上面的对象放在同一应用中，采用三种分别标注的关系：**Context 派生表示访问位置的继承；父子 Fiber 表示释放责任；服务依赖表示当前运行需要哪个提供者**。后两种关系可以连接同一批对象，却不是同一棵树。

![Cordis 根管理服务、Context 派生、装载归属与依赖关系](materials/objects.svg)

**沿 Context 看可见范围。** alphaCtx 派生自 app，插件收到的 ctx 又派生自装载入口。它们复用根部管理服务；workspaceName 来自普通属性继承，report/storage 的局部键来自 isolate。图省略连续 isolate/extend 的中间 Context 与代理，虚线表达派生关系，不声称是每一步直接 prototype。

**沿 Fiber 看谁负责释放。** alpha 的 workspace Fiber 管理 storage、report 和 batch 三类子装载。三个子插件是兄弟：report 对 storage 的依赖不会把 report 变成 storage 的子插件。根部格式和观察插件归根 Fiber 管理，所以关闭 alpha 不会关闭它们。beta 的结构相同。

**沿依赖看谁影响谁。** batch 声明 report；report 声明 storage。提供者变化会触发依赖方重新检查，方向从提供者传播到消费者；消费者声明依赖并不取得提供者的释放所有权。关闭 batch 本身不会因此关闭 report。

第一份 framework 快照可给结构填入具体身份：workspace 的 Fiber 为 5/6，storage 为 7/8，report 为 9/10，首批 batch 为 11/12；服务实例编号分别为 1/2。UID 来自框架的装载计数，instanceId 来自应用计数器。它们用于定位对象，理解图时先认角色与箭头，再按需核对数字。

这也把 2011–2013 连在一起：2011/2012 的 manual、scheduled 只是不同调用 Context，仍可指向同一业务实例；2013 额外使用不同隔离键，并分别执行提供者代码，才有两组服务。**Context 的数量、Runtime 的数量、Fiber 的数量和服务实例的数量，没有一一对应关系。**


## M2 从插件装载到一份报告返回

约 35 分钟。先看框架怎样让提供者与消费者进入可运行状态，再跟 alpha 的 lamp 报告。两段流程的连接点是插件 apply：Cordis 调度它，应用在其中注册能力或执行自己的业务。


### M2.1 装载流程：依赖就绪后，谁调用 apply

M1 中 workspacePlugin 先等待 storage，再安装 report。两个应用插件在 [report-service.ts](app/report-service.ts)、[report-batch.ts](app/report-batch.ts) 声明的依赖如下；这是结构摘录，apply 函数体已省略：

```ts
const reportProvider = {
    inject: ["storage"],
    apply(ctx, config) { /* new ReportService(ctx, config) */ },
};
const reportBatch = {
    inject: ["report"],
    async apply(ctx, config) { /* 调用服务、写文件、通知 */ },
};
```

类型声明让 TypeScript 知道 `ctx.report` 的类型；inject 告诉运行框架这次插件执行需要什么。安装 provider 时，真正的新建与登记发生在其 apply 内：`new ReportService()` 调用 [Service 构造器](materials/cordis/src/service.ts) 的 `ctx.reflect.provide(name, self, check)`，把实例登记为一个 Impl。Impl 是包含 value、提供方 Fiber 等信息的实现记录。

![从 Context 插件入口到依赖检查、apply 与服务发布的装载流程](materials/installation-flow.svg)

沿图依次定位 [Fiber 构造器、_checkImpl、_refresh、_reload](materials/cordis/src/fiber.ts)，只读下面这些连接：

1. Registry 找到 Runtime，创建 Fiber；Fiber 建立插件 ctx，保存配置、依赖声明与所属 Runtime。
2. `_checkImpl(name)` 通过 Reflect 按该 ctx 的隔离键查找**已 ACTIVE 的提供者**，把找到的 Impl 放入候选 `_store`。
3. Fiber 通过父级 effect 登记本次装载与释放动作，再由 `_refresh()` 根据依赖和更新版本决定运行目标。依赖齐备时，`_reload()` 复制候选表到本轮 `store`，在异步边界之后执行插件回调。
4. reportProvider.apply 创建服务，provide 写入登记；当前装载完成后 Fiber 转为 ACTIVE，框架再通知相关依赖方重新检查。reportBatch 才能绑定 report 并运行其 apply。

本例按提供者在前的顺序安装，因此正常通过这些阶段。依赖暂缺时，消费者 Fiber 可以已经存在，却仍处于 PENDING，尚未执行 apply。恢复后的同一检查链见 M3.3。`await plugin` 实际等待该 Fiber 的当前加载／卸载任务稳定，并抛出已保存的装载错误；在一般缺依赖场景中，返回本身不保证 ACTIVE。本文的正常路径之所以可继续，是因为所需提供者已经就绪。

**装载与业务调用的连接。** 本例把“一批报告”写在 reportBatch.apply 中，所以安装这个调用插件就会生成报告。这是 ReportDesk 的入口安排；日常调用 `reportView.generate()` 并不会自动创建一个新的 provider Fiber。提供者装载一次后，可以被反复调用。


### M2.2 同一次调用里的调用位置与定义位置

下面跟踪 [report-batch.ts](app/report-batch.ts) 的实际业务入口。这里的 ctx 是 batch Fiber 创建的 Context；job.input 来自 lamp 的文本、标题与 includeExcerpt 选项。

```ts
const reportView = ctx.report;
for (const job of config.jobs) {
    const markdown = reportView.generate(job.id, job.input);
    const fileId = `${ctx.workspaceName}-${job.id}`;
    await writeReport(fileId, markdown);
    ctx.emit("report/completed", { id: fileId, title: job.input.title });
}
```

`ctx.report` 通过 Reflect 的属性读取进入 batch 的本轮依赖表，取出 report Impl.value，再包装成关联当前调用 Context 的服务视图。真正的 ReportService 对象仍由提供者创建并登记；取视图、取方法都不等于重装插件或重新 new 服务。

[ReportService.generate](app/report-service.ts) 的关键部分如下，省略日志：

```ts
const workspaceName = this.ctx.workspaceName;
const markdown = this.ctx.waterfall("report/format", input, () => {
    return `工作区：${workspaceName}\n\n` + renderReport(input, this.config.excerptLength);
});
this.ctx.storage.save(id, markdown);
return markdown;
```

这里需要同时保存两种关系，正是 2012 方法代理与 shadow 的作用：

| 读取 | 应当从哪里取得 | 为什么 |
| --- | --- | --- |
| this.ctx.workspaceName | 当前调用处的 Context 继承链 | 同一个方法要知道这次从哪个入口调用 |
| this.config / this.instanceId | 原 ReportService 实例 | 配置与实例状态属于提供者创建的对象 |
| this.ctx.storage | report 定义处 Fiber 的本轮依赖记录 | batch 只声明 report，storage 是报告服务代码自己的依赖 |

在 [utils.ts 的 createTraceable → createShadow → createShadowMethod](materials/cordis/src/utils.ts) 中，读取普通方法会准备一个替代接收者：其 ctx 从调用处派生，同时保存 `symbols.shadow = defSite`，记录原服务的定义 Context。以 `reportView.generate(...)` 调用时，方法代理把匹配的 this 换成这个接收者；原服务对象的 ctx 没有被永久改写。

单独观察 workspaceName 或 2012 的 entryLabel，普通的调用方视图已能解释其值从哪里来。**shadow 进一步把“当前调用信息”和“方法代码的依赖归属”同时带进方法体。** 如果只把方法里的 ctx 当作 batch 的 ctx，就会丢失报告服务自己的 storage 依赖来源。

对应 [reflect.ts 的成功读取分支](materials/cordis/src/reflect.ts)，省略父级查找与错误处理：

```ts
const defSite = ctx[symbols.shadow] ?? ctx;
let fiber = defSite.fiber;
const impl = fiber.store?.[prop];
if (impl) return getTraceable(ctx, impl.value);
```

本次 prop 是 storage，defSite.fiber 是 alpha 的 report Fiber 9；`store.storage` 指向 storage Fiber 7 提供的 Impl。调用信息继续来自 batch 的入口，依赖使用 report 本轮已经绑定的实现。beta 走相同机制，但绑定另一组实例。读到成功返回即可；本课复用这条已学路径，不要求再次逐层追完所有代理分支。


### M2.3 业务时序：直接调用、环绕处理与通知各做什么

[format-plugins.ts](app/format-plugins.ts) 用 `ctx.on("report/format", callback)` 先登记 quoteFormat，再登记 footerFormat；通知和统计通过 [notificationPlugin](app/notification-plugin.ts)、[statisticsPlugin](app/statistics-plugin.ts) 登记 report/completed。EventsService 在 `_hooks` 中保存回调及其登记 ctx，并把撤销登记的动作接入该插件 Fiber 的资源清理。

generate 调用 waterfall 时，input 传给每个格式回调，最后一个参数是渲染主体的终点函数；框架为各层提供 next。两层回调均先调用 next，再加工其结果，所以进入顺序为 quote → footer → render，返回加工顺序为 render → footer → quote。这一层叠关系来自回调代码与 waterfall 的返回传递，而非插件之间互相导入。

![一份输入经过服务、格式扩展、内存与磁盘保存、完成观察的时序](materials/call-sequence.svg)

| 连接方式 | 谁发起、传什么 | 本例结果与等待边界 |
| --- | --- | --- |
| 普通服务方法 | batch 调用 generate(id, input)，report 调用 storage.save(id, markdown) | 都是同步方法；generate 返回字符串，save 写入内存 Map |
| waterfall | generate 传 input 与渲染终点，格式回调调用 next | 返回值逐层构成最终报告；本例回调同步 |
| await 应用函数 | batch 把字符串交给 writeReport | 等文件写出后才继续；它是应用文件操作，不是 Cordis 的服务调度 |
| emit | batch 发送 report/completed 的 id、title | 本例两个同步观察者当场执行；emit 不收集它们的结果，也不等待它们返回的 Promise |

因此一次报告有两个不同保存位置：storage.save 保存内存数据，writeReport 保存磁盘文件。完成通知由 batch 在文件成功写出后发出，generate 自身没有发出这个事件。reportBatch.apply 的 Promise 随这批业务完成而兑现，它所属 Fiber 的装载与资源归属仍保留，见 M3.2。

从系列视角看，2006 的完成通知与 2009 的格式环绕都复用 EventsService，却有不同的数据与返回约定。框架内部也使用事件系统，例如配置处理经过 internal/config；业务仍通过明确的服务方法完成生成与保存。理解框架时应沿每个实际触发处判断输入、this、返回值和等待关系，不能仅凭它们都叫“事件”推断行为相同。阅读 [events.ts 的 on/register、waterfall、emit](materials/cordis/src/events.ts) 到上述回调执行和返回即可。


### M2.4 按需在 VS Code 跟一条调用

打开本课目录，选择 **Cordis 2014 · 01 · 完整流程与对象图解** 调试配置。源码入口、cwd 与 Notebook 相同。先在 [report-batch.ts](app/report-batch.ts) 的 `reportView.generate(...)` 行设断点，观察 `ctx.workspaceName`，再进入本课 [report-service.ts](app/report-service.ts)。

若代理层单步过多，可直接在 generate 首行加 `this.ctx.workspaceName === "alpha"` 条件断点，观察 this.instanceId 与 this.ctx.storage.instanceId 均为 1，走到 storage.save 后停止。需要补读定义处时，再定位本课 reflect.ts 的 `fiber.store?.[prop]`。不要求逐层跟完整个 utils.ts。

行号以当前源文件为准。


## M3 把数据位置与生命周期连起来

约 40 分钟。M1 的结构说明“谁关联谁”，M2 说明“怎样执行”，本段进一步说明“关系保存在哪里，以及何时改变”。把一个业务请求的结束、一个 apply 周期的结束和一次插件装载的结束分开，才能理解恢复、更新与关闭。


### M3.1 从对象图找到框架真正维护的数据

先看服务查找的三步：Context 的有效隔离映射为服务名选 Symbol；共享 ReflectService.store 用这个 Symbol 找 Impl；消费者 Fiber 在依赖准备时将 Impl 绑定到本轮 store。映射、全局登记和本轮依赖是三个位置，分别回答**哪个名字空间、哪份实现、这轮实际使用谁**。

[provide（reflect.ts）](materials/cordis/src/reflect.ts) 中的关键写入为：

```ts
this.ctx.root[symbols.isolate][name] ??= Symbol(name);
const key = this.ctx[symbols.isolate][name];
const impl: Impl = { name, value, fiber: this.ctx.fiber, check };
this.store[key] = impl;
this.ctx.fiber.store![name] = impl;
```

第一行只在根默认映射缺失时补一个默认 Symbol；实际登记使用第二行从**当前 Context**读出的键。alpha 的局部 Symbol 与根默认 Symbol 可以描述相同，却不是同一个对象。根有名字映射，不表示根键下已有实例。本例四次快照中 rootHasReport、rootHasStorage 均为 false，而局部服务正常可用。

![服务名映射、共享实现登记、消费者候选依赖与本轮绑定](materials/data-map.svg)

[Fiber._checkImpl 与 _reload](materials/cordis/src/fiber.ts) 中，`_store` 是随依赖检查更新的候选表，`store = { ...this._store }` 是本轮执行使用的表。表的浅复制保留对同一 Impl 的引用；没有复制业务服务实例。运行时 `this.ctx.storage` 读取本轮表，环境变化则更新候选表并决定是否结束旧周期。

将这条服务路径放回整个框架，可以看到不同数据有不同的维护者：

| 数据位置 | 保存的关系 | 何时写入／移除 |
| --- | --- | --- |
| registry._internal / Runtime.fibers | callback → Runtime → 多次装载 | plugin 登记；dispose 移除 Fiber，最后一条装载移除后删除对应 Runtime |
| ctx[symbols.isolate] | 服务名 → 有效 Symbol 键 | isolate 派生局部覆盖；provide 按需补根默认键 |
| reflect.store | Symbol → Impl，含 value 与提供方 fiber | provide 登记；相应释放函数撤下 Impl |
| Fiber._store / Fiber.store | 依赖名 → 候选／本轮 Impl；store 也记录本轮自己提供的服务 | 检查依赖、开始 apply 周期时写入；卸载后本轮 store 置空 |
| events._hooks | 事件名 → 回调、登记 ctx 与选项 | on/register 添加；调用 on 返回的释放函数，或所属周期清理时撤销 |
| Fiber._disposables | 这一轮已经登记的释放动作 | effect 和 apply 返回值等加入；_unload 执行并清空 |
| StorageService.records / 报告文件 | 应用自己的内存结果／磁盘产物 | save／writeReport 写入；生命周期语义由应用定义 |

**服务解除登记、JavaScript 对象被回收、业务数据被删除是不同的事情。** 框架清理的是已登记的关系和释放动作；一个旧服务仍可能被其他变量引用，垃圾回收也不等于删除报告文件。表中给出的数据位置能解释“看不到服务了”具体发生在哪一层。


### M3.2 调用结束、插件关闭与资源范围

先看本课同一次运行的快照。数量只统计表头指定的应用记录，不包含框架内部服务与监听。

| 时刻 | 局部 report/storage 登记项 | workspacePlugin 的 Fiber 数 | reportBatch 的 Fiber 数 | report/format 监听 | report/completed 监听 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 两组各写一份后 | 4 | 2 | 2 | 2 | 2 |
| 关闭 alpha 后 | 2 | 1 | 1 | 2 | 2 |
| beta 再写一份后 | 2 | 1 | 2 | 2 | 2 |
| 关闭 beta 及四个共享插件后 | 0 | 0 | 0 | 0 | 0 |

第一行已经完成两批业务，但两个 batch Fiber 仍在 Runtime 中。第二次给 beta 安装 batch 时，又增加一条装载记录；这次调用的 report/storage 仍是原实例 2，内存由 lamp 增为 lamp、bottle。**函数返回或 Promise 兑现，表示这次 apply 执行完成；自动卸载并不是它的含义。** 完成的插件仍可保留服务、监听和子插件，直到所属周期清理或主动 dispose。

关闭 alpha 时，M1 的三种关系分别起作用：

1. **释放归属**：workspace Fiber 中登记了子插件 dispose，组关闭会释放 alpha 的 storage、report、batch 装载；这些子 Fiber 从各自 Runtime.fibers 移除。
2. **依赖关系**：服务的释放函数从 reflect.store 删除相应 Impl，notify 直接调用受影响消费者的 `_checkImpl/_refresh`。它随后也派发 internal/service；依赖刷新并不是仅靠这条事件间接发生。
3. **可见范围**：notify 的默认筛选比较服务名对应的隔离 Symbol。alpha 的服务变化不要求 beta 重选 storage，beta 的原服务和内存数据得以继续使用。

根部四个格式／观察插件不属于 alpha 的子装载，所以两类业务监听继续为 2。它们在 finally 中各自 dispose 后才归零。服务名隔离也不自动把所有业务事件隔离：本例两组共用格式与完成观察者。

对应阅读 [Fiber 构造器中的父级 effect 与 _unload](materials/cordis/src/fiber.ts)、[ReflectService.provide 的释放函数与 notify](materials/cordis/src/reflect.ts)，看到“移除装载记录、撤下对应登记项、刷新相关依赖”就停止。兄弟资源可能并发清理，图不规定固定的兄弟先后。最后一行表示选列的应用插件已关闭，根 Context 与框架内部设施并不因此被宣称已销毁，三份报告文件也仍保留。


### M3.3 生命周期：同一套调度怎样处理三种变化

把前课的依赖恢复（2008）、配置更新（2010）与本课的关闭放回同一模型：**一个 Fiber 可以经历多轮 apply；每轮拥有当时的依赖绑定和资源，切换周期先释放旧资源，再按新的运行目标决定是否加载。** 最终 dispose 则结束这次装载本身。

![插件从待就绪到活动，以及依赖、配置与关闭触发的不同生命周期](materials/lifecycle.svg)

图中正常装载与最终关闭对应本课 01 的运行；依赖撤下／恢复、配置更新两条支线是已学固定源码的归纳，本课 01 未执行它们。“旧／新实例”描述源码中的创建关系，不冒充本次测得的实例编号。

**先读公共路径。** [Fiber._refresh](materials/cordis/src/fiber.ts) 根据 `_revision` 与所依赖提供方的 uid 形成当前运行目标，缺必需依赖时目标为内部标记 INACTIVE。`_setEpoch()` 比较前后目标，通过 inertia（当前加载／卸载过程的 Promise）串联加载与卸载；`_unload()` 执行本轮释放动作、清空 store，随后按目标转入等待或下一轮 `_reload()`。PENDING、LOADING、ACTIVE、UNLOADING、DISPOSED 是 Fiber 的阶段；INACTIVE 是运行目标的内部标记。

| 触发 | report Fiber 是否保留 | 本轮 report 服务与资源 | 后续如何继续 |
| --- | --- | --- | --- |
| 一次 generate 返回 | 保留，仍可调用 | 保留；只是本次业务调用结束 | 同一实例可以处理下一份输入 |
| storage 暂缺 | 正常依赖停用路径保留装载，卸载后 PENDING | 旧 report 登记和本轮资源被释放 | 同隔离范围的 storage 就绪后重新检查、加载，新建 ReportService |
| report Fiber.update(config) | 本例默认 restart 路径保留 Fiber | 保存配置、revision 改变，清理旧周期后重跑 apply | 新 ReportService 使用新配置；未撤下的 storage 可保持原对象 |
| report Fiber.dispose() | 结束该次装载，uid 置 null，并移出 Runtime | 清理最后一轮资源；进入 DISPOSED | 要再次装载需新的 plugin 调用，产生新的 Fiber |

**依赖恢复沿依赖关系传播。** storage 撤下后，reportProvider 因依赖缺失停用；report 登记的撤下又影响依赖 report 的 batch。storage 恢复使 report 重新就绪，仍装载的 batch 也可能再次执行 apply。这不是父子插件逐层关闭，而是提供者状态变化影响消费者。源码停止点为 `notify → _checkImpl → _refresh → _unload/_reload`。

**配置更新复用同一调度。** update 保存配置、清除已保存错误，经 internal/update 进入本例的默认 restart；revision 变化使旧周期退出、新周期开始。若另有更新监听，可能改变是否重启；本例没有这种拦截。源码停止点为 [update/restart](materials/cordis/src/fiber.ts) 回到上述公共路径，不再重读所有异步分支。

**释放动作是生命周期与实际资源的连接点。** [ctx.effect 对应 Fiber.effect](materials/cordis/src/fiber.ts)：创建服务登记、监听、子插件时，就将撤下它们的函数归入当前周期；插件 apply 自己返回的清理函数也会被收集。框架能统一释放这些登记资源，是因为创建时已经记录了清理责任；一般的文件写入不会凭空获得删除动作。

应用仍决定数据保留和业务重跑的意义。新的内存 storage 没有自动迁移旧 Map 的数据；仍装载的 batch 重新执行其 jobs，也不等于框架提供了业务请求的可靠重放。这里归纳正常依赖变化与默认配置更新，不能把它们推成所有执行错误都会自动恢复。


## M4 从 ReportDesk 回到整个 Cordis 系列

约 10 分钟。本段把图里的结构对应回前课，将具体变量换成框架职责后，仍能解释它们怎样协作。它是阅读归纳，不是额外复述或验收题。


### M4.1 前面的课程怎样汇合到本课

下表用于回查卡点，正文已经给出本课所需的关系；无需逐课重读。

| 已学路径 | 在本课中的汇合点 | 放到其他 Cordis 应用仍成立的关系 |
| --- | --- | --- |
| [2001 插件入口](../2001.Cordis-Context-and-Function-Plugins/LearningTask.ipynb)、[2002 根装配与方法转发](../2002.Cordis-Root-Context-and-Method-Dispatch/LearningTask.ipynb)、[2003 的 Context 派生](../2003.Cordis-Context-Views-and-Method-Binding/LearningTask.ipynb) | M1 的根对象与当前调用位置 | Context 组织访问与元信息，管理服务保存共享状态；派生 Context 不自动创建一套服务 |
| [2004 首个业务插件](../2004.Cordis-ReportDesk-First-Plugin/LearningTask.ipynb)、[2005 可复用服务](../2005.Cordis-ReportDesk-Reusable-Service/LearningTask.ipynb) | M2 的装载与服务发布 | 插件执行应用代码，服务对外提供能力；声明类型、声明依赖、创建实例和登记服务是不同动作 |
| [2006 事件观察](../2006.Cordis-ReportDesk-Events-and-Observers/LearningTask.ipynb)、[2009 格式环绕](../2009.Cordis-ReportDesk-Format-Plugins/LearningTask.ipynb) | M2 的通知与 waterfall | 回调可独立登记；派发方式和回调代码共同决定结果、顺序与等待 |
| [2007 卸载与重装](../2007.Cordis-ReportDesk-Observer-Lifecycle/LearningTask.ipynb)、[2008 依赖停用与恢复](../2008.Cordis-ReportDesk-Storage-Dependency/LearningTask.ipynb)、[2010 配置更新](../2010.Cordis-ReportDesk-Config-Update/LearningTask.ipynb) | M3 的周期与释放责任 | 依赖、配置、最终关闭通过同一 Fiber 调度连接到资源清理，但保留的对象不同 |
| [2011 共享服务视图](../2011.Cordis-ReportDesk-Service-Views/LearningTask.ipynb)、[2012 方法与依赖上下文](../2012.Cordis-ReportDesk-Method-Context/LearningTask.ipynb) | M2.2 的调用处与定义处 | 共享实例可服务多个调用入口，方法同时需要调用信息与自身依赖归属 |
| [2013 同名服务隔离](../2013.Cordis-ReportDesk-Workspace-Isolation/LearningTask.ipynb) | M1 的两组实例、M3 的登记与通知筛选 | 局部名字映射选择实现；隔离映射、实例创建、资源归属需要一起看 |

这条主线从“如何使用插件”逐步走到“框架如何维持关系”。ReportDesk 中的报告、存储、格式和观察者，分别让服务协作、依赖变化、扩展回调和资源清理有了可追踪的具体对象。


### M4.2 用结构、时序与变化共同解释框架

**先从入口定位当前 Context 和 Fiber。** `ctx.plugin()`、服务方法、事件派发是不同入口；它们各自携带参数、this 和当前访问位置。不要把 Context、插件代码、装载记录与业务实例都统称为“插件对象”。

**再找真正保存关系的数据。** Runtime 归类同一回调的装载，Reflect 登记服务实现，Fiber 绑定本轮依赖，Events 保存监听。看见一个方法名或一次打印之后，要能回到保存这份关系的字段，以及写入它的操作。

**随后把正常调用接到变化后的行为。** 调用只使用现有服务；依赖或配置变化可能结束当前 apply 周期；dispose 结束装载。释放责任沿父子 Fiber 管理，服务变化沿依赖关系传播，可见实现由 Context 的映射选择。三种关系共同解释行为，不必把整个框架想成一条继承链或一棵插件树。

这就是本课的系列收束：**Cordis 让可组合的插件在各自 Context 中注册能力、声明依赖和扩展点，并把运行阶段与资源释放连接起来；ReportDesk 的应用代码在这些关系之上完成报告业务。** 五张图分别提供对象结构、装载流程、业务时序、登记数据和生命周期视角，沿同一组对象可以互相对照。

自由笔记可只记录仍未接上的关系。


# 本课收束

Context 派生、Fiber 释放归属与服务依赖是三种关系；业务调用结束、apply 周期切换与最终卸载是三种生命周期边界。将对象关系、数据存储位置和触发流程结合起来，才能解释恢复、更新和关闭时的保留与替换。
